# 🤖 SmolVLA GPU 訓練與雲端硬碟自動備份 / 接續訓練筆記本

本筆記本專門為 **Google Colab GPU（T4 / V100 / A100）** 環境設計，支援：
1. **開頭選擇**：從頭開始訓練 (Scratch) 或從 Google 雲端硬碟 `/content/data/SmolVLA_training/` 讀取上次權重接續訓練 (Resume)。
2. **即時日誌**：訓練與評估進度逐行即時串流輸出，一目了然。
3. **模擬評估**：自動載入最新 Checkpoint 進行 Aloha 抓取推論並播放生成的 `.mp4` 影片。
4. **最後備份**：自動將**模型權重、參數、評估影片、訓練方法說明與本筆記本**完整存入 Google 雲端硬碟 `/content/data/SmolVLA_training/`。

⚠️ **請先確認 GPU 已啟用**：
點擊選單 **執行階段 (Runtime) -> 變更執行階段類型 (Change runtime type) -> 硬體加速器選擇 T4 GPU**。

In [ ]:
# 1. 驗證 GPU 狀態與環境配置
import torch
import os

if not torch.cuda.is_available():
    raise SystemError("❌ 未檢測到 GPU！請至選單 [執行階段 -> 變更執行階段類型] 切換至 T4 GPU 後再執行。")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print("✅ GPU 檢測成功！")
print("顯卡型號:", gpu_name)
print("顯存大小:", round(vram_gb, 2), "GB")

# 獲取 / 同步最新 LeRobot 倉庫代碼
if not os.path.exists("/content/lerobot"):
    os.system("git clone https://github.com/Child-pi/lerobot.git")
else:
    os.chdir("/content/lerobot")
    os.system("git pull origin main")

%cd /content/lerobot


### 📁 步驟 2：Google 雲端硬碟設定與訓練模式選擇
您可在此選擇是**從頭開始訓練 (Scratch)**，或是**從 Google 雲端硬碟讀取上次訓練的資料繼續微調訓練 (Resume)**。

In [ ]:
#@title 訓練模式與雲端硬碟設定 (Training Mode & Google Drive Setup)
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
TRAINING_MODE = "從頭開始訓練 (Scratch)" #@param ["從頭開始訓練 (Scratch)", "接續上次訓練 (Resume / Continue)"]
DRIVE_BACKUP_DIR = "/content/data/SmolVLA_training" #@param {type:"string"}

import os
import shutil
import glob

RESUME_CHECKPOINT_PATH = None

# 1. 掛載 Google Drive 並對接目錄
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("⏳ 正在掛載 Google 雲端硬碟...")
            drive.mount("/content/drive")
        print("✅ Google Drive 掛載成功！")
        
        MY_DRIVE_TARGET = "/content/drive/MyDrive/SmolVLA_training"
        os.makedirs(MY_DRIVE_TARGET, exist_ok=True)
        os.makedirs("/content/data", exist_ok=True)
        
        if not os.path.exists(DRIVE_BACKUP_DIR):
            try:
                os.symlink(MY_DRIVE_TARGET, DRIVE_BACKUP_DIR)
                print("🔗 已建立符號連結:", DRIVE_BACKUP_DIR, "->", MY_DRIVE_TARGET)
            except Exception:
                os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    except Exception as e:
        print("ℹ️ 提示 (非 Colab 或跳過 Drive 掛載):", e)
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
else:
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. 判斷訓練模式
if "接續" in TRAINING_MODE:
    print("🔍 正在從雲端硬碟中搜尋上次訓練資料:", DRIVE_BACKUP_DIR)
    candidate_patterns = [
        os.path.join(DRIVE_BACKUP_DIR, "outputs/train/**/pretrained_model"),
        os.path.join(DRIVE_BACKUP_DIR, "train/**/pretrained_model"),
        os.path.join(DRIVE_BACKUP_DIR, "checkpoints/**/pretrained_model"),
        os.path.join(DRIVE_BACKUP_DIR, "pretrained_model"),
        os.path.join(DRIVE_BACKUP_DIR, "**/*pretrained_model*"),
    ]
    found_ckpts = []
    for pat in candidate_patterns:
        found_ckpts.extend(glob.glob(pat, recursive=True))
    found_ckpts = [p for p in found_ckpts if os.path.isdir(p) and (os.path.exists(os.path.join(p, "config.json")) or os.path.exists(os.path.join(p, "model.safetensors")))]
    found_ckpts = list(dict.fromkeys(found_ckpts))
    
    if found_ckpts:
        found_ckpts.sort(key=os.path.getmtime)
        RESUME_CHECKPOINT_PATH = found_ckpts[-1]
        print("🎉 成功讀取上次訓練的權重 Checkpoint:", RESUME_CHECKPOINT_PATH)
        print("👉 本次訓練將從此 Checkpoint 開始接續微調！")
        
        # 若有完整的歷史輸出目錄，同步一份至本機
        src_train_dir = os.path.join(DRIVE_BACKUP_DIR, "outputs/train/smolvla_aloha")
        dst_train_dir = "/content/lerobot/outputs/train/smolvla_aloha"
        if os.path.exists(src_train_dir) and not os.path.exists(dst_train_dir):
            try:
                shutil.copytree(src_train_dir, dst_train_dir, dirs_exist_ok=True)
                print("📁 已同步歷史輸出檔案至本機:", dst_train_dir)
            except Exception as err:
                print("ℹ️ 同步歷史檔案提示:", err)
    else:
        print("⚠️ 在雲端硬碟中尚未找到上次的訓練權重！")
        print("👉 將自動切換為【從頭開始訓練 (Scratch)】模式。")
        RESUME_CHECKPOINT_PATH = None
else:
    print("🆕 已選擇【從頭開始訓練 (Scratch)】模式。")
    print("👉 將使用預訓練基礎模型 lerobot/smolvla_base 進行全新訓練。")
    RESUME_CHECKPOINT_PATH = None


### 📦 步驟 3：安裝 LeRobot 及其依賴項

In [ ]:
%cd /content/lerobot

print("📦 安裝 LeRobot 與 smolvla 依賴...")
!pip install -e ".[smolvla]"

print("📦 安裝 Aloha 模擬環境與影片編碼套件...")
!pip install dm_env dm_tree glfw lxml "mujoco>=3.0.0,<3.9.0" pyopengl pyparsing
!pip install dm-control --no-deps
!pip install gym-aloha --no-deps
!pip install -q av

print("🎉 環境安裝完成！")


### 🔑 步驟 4：GitHub 認證設定（選填，用於推送到 GitHub 倉庫）

In [ ]:
import os
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    if GITHUB_TOKEN:
        GITHUB_USER = "Child-pi"
        REPO_NAME = "lerobot"
        os.system("git config --global user.email nokiya@gmail.com")
        os.system("git config --global user.name Child-pi")
        remote_url = "https://" + GITHUB_USER + ":" + GITHUB_TOKEN + "@github.com/" + GITHUB_USER + "/" + REPO_NAME + ".git"
        os.system("git remote set-url origin " + remote_url)
        print("✅ GitHub 認證設定成功！")
    else:
        print("ℹ️ 未設定 GITHUB_TOKEN，略過 GitHub 認證（不影響本機訓練與評估）。")
except Exception as e:
    print("ℹ️ 略過 GitHub 認證:", e)


### 🚀 步驟 5：啟動 GPU 訓練 SmolVLA
在 Aloha 模擬機械手臂夾取資料集（`aloha_sim_insertion_human`）上進行 500 步離線微調。
系統會依據步驟 2 自動切換**從頭訓練**或**接續訓練**，日誌以**即時串流**輸出。

In [ ]:
import os
import shutil
import subprocess
import torch

if os.path.exists("/content/lerobot"):
    os.chdir("/content/lerobot")
    os.system("git pull origin main")

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"

if torch.cuda.is_available():
    torch.cuda.empty_cache()

output_dir = "outputs/train/smolvla_aloha"

# 判定起始權重：接續訓練 vs 從頭訓練
if "RESUME_CHECKPOINT_PATH" in globals() and RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
    policy_path = RESUME_CHECKPOINT_PATH
    print("🔁 【接續訓練模式】使用上次保存的權重作為起點:", policy_path)
else:
    policy_path = "lerobot/smolvla_base"
    print("🆕 【從頭訓練模式】使用基礎預訓練模型:", policy_path)
    if os.path.exists(output_dir):
        print("🧹 清理先前的輸出目錄:", output_dir)
        shutil.rmtree(output_dir)

train_cmd = [
    "lerobot-train",
    "--policy.path=" + policy_path,
    "--policy.push_to_hub=false",
    "--dataset.repo_id=lerobot/aloha_sim_insertion_human",
    "--batch_size=2",
    "--steps=500",
    "--save_freq=500",
    "--log_freq=5",
    "--output_dir=" + output_dir,
    "--policy.empty_cameras=2",
    '--rename_map={"observation.images.top": "observation.images.camera1"}',
    "--wandb.enable=false"
]

print("🚀 開始執行 SmolVLA GPU 訓練（即時日誌輸出）...")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()

if proc.returncode == 0:
    print()
    print("🎉 SmolVLA GPU 訓練順利完成！")
else:
    print()
    print("❌ 訓練異常結束，狀態碼:", proc.returncode)


### 🎬 步驟 6：模擬評估與機械手臂抓取影片錄製
自動辨識訓練完成的 Checkpoint，在 Aloha 模擬器中執行推論並錄製 `.mp4` 影片。

In [ ]:
import os
import glob
import subprocess
import torch

if os.path.exists("/content/lerobot"):
    os.chdir("/content/lerobot")

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
if torch.cuda.is_available():
    torch.cuda.empty_cache()

candidate_patterns = [
    "/content/lerobot/outputs/train/**/pretrained_model",
    "outputs/train/**/pretrained_model",
    "/content/lerobot/outputs/train/**/checkpoints/*",
]
checkpoints = []
for pat in candidate_patterns:
    checkpoints.extend(glob.glob(pat, recursive=True))
checkpoints = [p for p in checkpoints if os.path.isdir(p) and (os.path.exists(os.path.join(p, "config.json")) or "pretrained_model" in p)]
checkpoints = list(dict.fromkeys(checkpoints))

if not checkpoints:
    print("❌ 找不到訓練好的模型，請確認上一格訓練是否完成。")
else:
    ckpt_path = checkpoints[0]
    print("✅ 找到評估目標模型:", ckpt_path)
    print("🎬 開始執行模擬評估並錄製影片...")
    
    eval_cmd = [
        "lerobot-eval",
        "--policy.path=" + str(ckpt_path),
        "--env.type=aloha",
        "--eval.n_episodes=1",
        "--eval.batch_size=1",
        "--eval.use_async_envs=false",
        '--rename_map={"observation.images.top": "observation.images.camera1"}',
        "--output_dir=/content/lerobot/outputs/eval/smolvla_aloha"
    ]
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    eval_proc = subprocess.Popen(eval_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(eval_proc.stdout.readline, ""):
        print(line, end="", flush=True)
    eval_proc.wait()
    
    if eval_proc.returncode == 0:
        print()
        print("✅ 模擬評估與錄影完成！請執行下一格播放影片。")
    else:
        print()
        print("⚠️ 評估指令結束，狀態碼:", eval_proc.returncode)


### 📺 步驟 7：播放機械手臂模擬抓取影片

In [ ]:
from IPython.display import Video, display
import glob
import os

if os.path.exists("/content/lerobot"):
    os.chdir("/content/lerobot")

search_patterns = [
    "/content/lerobot/outputs/eval/**/videos/*.mp4",
    "/content/lerobot/outputs/**/videos/*.mp4",
    "outputs/eval/**/videos/*.mp4",
    "outputs/**/videos/*.mp4",
    "/content/**/*.mp4"
]
all_videos = []
for pat in search_patterns:
    all_videos.extend(glob.glob(pat, recursive=True))
all_videos = list(dict.fromkeys(all_videos))
valid_videos = [v for v in all_videos if os.path.isfile(v)]

if valid_videos:
    valid_videos.sort(key=os.path.getmtime)
    latest_video = valid_videos[-1]
    print("🎉 成功找到機械手臂評估影片:", latest_video)
    print("檔案大小:", round(os.path.getsize(latest_video) / 1024, 1), "KB")
    display(Video(latest_video, embed=True, width=640))
else:
    print("⚠️ 尚未找到錄製的 .mp4 影片！請先確認上方評估儲存格是否已執行完畢。")


### 💾 步驟 8：將完整訓練資料儲存至 Google 雲端硬碟 (`/content/data/SmolVLA_training/`)
自動將本次訓練與評估的所有關鍵資產完整備份至 Google Drive：
- 📦 **模型權重 (Weights)**：Checkpoints、`model.safetensors`
- ⚙️ **訓練與模型參數 (Parameters & Configs)**：`config.json`、`train_config.json`
- 🎥 **模擬評估結果與影片 (Results & MP4 Videos)**：動作錄影與評估指標
- 📝 **訓練方法說明文件 (Training Methods)**：自動產生 `TRAINING_METHOD.md`
- 📓 **Google Colab 筆記本檔案 (.ipynb)**：備份當前執行的 Notebook

In [ ]:
#@title 執行資料備份至 Google Drive (Backup all training assets)
import os
import shutil
import datetime
import glob

BACKUP_DIR = "/content/data/SmolVLA_training"

# 1. 確保 Google Drive 已掛載並對接
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("⏳ 正在掛載 Google 雲端硬碟...")
        drive.mount("/content/drive")
    MY_DRIVE_TARGET = "/content/drive/MyDrive/SmolVLA_training"
    os.makedirs(MY_DRIVE_TARGET, exist_ok=True)
    os.makedirs("/content/data", exist_ok=True)
    if not os.path.exists(BACKUP_DIR):
        try:
            os.symlink(MY_DRIVE_TARGET, BACKUP_DIR)
        except Exception:
            os.makedirs(BACKUP_DIR, exist_ok=True)
except Exception as e:
    print("ℹ️ 提示 (非 Colab 或無 Drive 掛載):", e)
    os.makedirs(BACKUP_DIR, exist_ok=True)

print("📁 開始儲存訓練資料至:", BACKUP_DIR)

# 2. 備份權重與訓練參數 (Weights & Configs)
train_output_src = "/content/lerobot/outputs/train/smolvla_aloha"
train_output_dst = os.path.join(BACKUP_DIR, "outputs/train/smolvla_aloha")
if os.path.exists(train_output_src):
    print("📦 正在備份模型權重與參數 (Checkpoints / Weights / Configs)... Google Drive 同步中...")
    shutil.copytree(train_output_src, train_output_dst, dirs_exist_ok=True)
    print("✅ 權重與參數備份完成！")
else:
    print("⚠️ 找不到訓練輸出目錄:", train_output_src)

# 3. 備份評估結果與影片 (Evaluation Results & Videos)
eval_output_src = "/content/lerobot/outputs/eval/smolvla_aloha"
eval_output_dst = os.path.join(BACKUP_DIR, "outputs/eval/smolvla_aloha")
if os.path.exists(eval_output_src):
    print("🎥 正在備份評估結果與抓取影片 (Videos & Metrics)... Google Drive 同步中...")
    shutil.copytree(eval_output_src, eval_output_dst, dirs_exist_ok=True)
    print("✅ 評估結果與影片備份完成！")
else:
    mp4_files = glob.glob("/content/lerobot/outputs/**/*.mp4", recursive=True) + glob.glob("/content/**/*.mp4", recursive=True)
    if mp4_files:
        video_dst_dir = os.path.join(BACKUP_DIR, "videos")
        os.makedirs(video_dst_dir, exist_ok=True)
        for vf in set(mp4_files):
            if os.path.isfile(vf):
                shutil.copy2(vf, video_dst_dir)
        print("✅ 已備份生成的 mp4 影片至:", video_dst_dir)

# 4. 備份 Google Colab 筆記本檔案 (Notebook .ipynb)
notebook_candidates = [
    "/content/lerobot/train_smolvla_gpu.ipynb",
    "/content/lerobot/train_smolvla.ipynb",
    "/content/train_smolvla_gpu.ipynb",
    "/content/train_smolvla.ipynb",
]
saved_nb = None
for nb_path in notebook_candidates:
    if os.path.exists(nb_path):
        dst_nb = os.path.join(BACKUP_DIR, os.path.basename(nb_path))
        shutil.copy2(nb_path, dst_nb)
        saved_nb = dst_nb
        break
if saved_nb:
    print("📓 已備份 Google Colab 筆記本:", saved_nb)
else:
    print("ℹ️ 未在本地找到 .ipynb 檔案，跳過筆記本備份。")

# 5. 生成訓練方法與參數說明文件 (Training Methods Documentation)
now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
readme_content = f"""# SmolVLA 訓練方法與執行紀錄 (Training Record)

- **紀錄時間**: {now_str}
- **模型架構**: SmolVLA (lerobot/smolvla_base)
- **訓練資料集**: lerobot/aloha_sim_insertion_human (Aloha 模擬機械手臂雙臂夾取)
- **訓練步數 (Steps)**: 500 steps
- **批次大小 (Batch Size)**: 2
- **優化器與學習率**: 預設 Auto-scaled LR scheduler (warmup: 16, decay: 500)
- **鏡頭映射 (Rename Map)**: observation.images.top -> observation.images.camera1
- **虛擬相機數量 (Empty Cameras)**: 2 (補充左/右腕相機)
- **模擬環境**: Aloha (gym_aloha / MuJoCo EGL Headless)
- **硬體環境**: Google Colab GPU (CUDA)
- **備份路徑**: {BACKUP_DIR}

## 目錄結構說明:
- `outputs/train/smolvla_aloha/`: 模型權重 (checkpoints, model.safetensors, config.json)
- `outputs/eval/smolvla_aloha/`: 評估指標、日誌與錄製的機械手臂動作影片 (.mp4)
- `train_smolvla_gpu.ipynb`: 執行此訓練所使用的 Google Colab 筆記本
- `TRAINING_METHOD.md`: 本份訓練方法與參數說明文件
"""

readme_path = os.path.join(BACKUP_DIR, "TRAINING_METHOD.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)
print("📝 已生成訓練方法說明文件:", readme_path)

# 若有實體 MyDrive 目錄且非同一目錄，確保內容完整寫入
if os.path.exists("/content/drive/MyDrive/SmolVLA_training") and not os.path.samefile(BACKUP_DIR, "/content/drive/MyDrive/SmolVLA_training"):
    try:
        shutil.copytree(BACKUP_DIR, "/content/drive/MyDrive/SmolVLA_training", dirs_exist_ok=True)
        print("☁️ 已同步確認實體檔案存入 Google Drive: /content/drive/MyDrive/SmolVLA_training/")
    except Exception as e:
        pass

print("\n🎉🎉 全部訓練資料已成功儲存至 Google 雲端硬碟:")
print("   📍", BACKUP_DIR)
for root, dirs, files in os.walk(BACKUP_DIR):
    level = root.replace(BACKUP_DIR, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files[:6]:
        print(f"{subindent}📄 {f}")
    if len(files) > 6:
        print(f"{subindent}... (共 {len(files)} 個檔案)")
